# Chapitre 1 — La Series

## Le problème

Un tableau NumPy repère ses valeurs uniquement par leur **position**. Pour suivre le stock de quelques produits, il faut se souvenir que la position 0 correspond au clavier, la position 1 à la souris, etc. Si deux tableaux ne listent pas les produits dans le même ordre, les additionner donne un résultat faux, sans aucune erreur.

La **Series** ajoute à un tableau NumPy une **étiquette par valeur** : l'index. Les opérations entre Series s'effectuent alors étiquette par étiquette, et non position par position.

In [1]:
import numpy as np
import pandas as pd

## 1. Créer une Series

### À partir d'une liste

In [4]:
s = pd.Series([0.25, 0.5, 0.75, 1.0])
s

0    0.25
1    0.50
2    0.75
3    1.00
dtype: float64

Trois éléments s'affichent : l'**index** à gauche (créé automatiquement : 0, 1, 2, 3), les **valeurs** à droite, et le **type** en dessous (`dtype: float64`).

### Avec un index explicite et un type imposé

In [ ]:
s = pd.Series([0.25, 0.5, 0.75, 1.0], index=["a", "b", "c", "d"], dtype=np.float32)
s

Le paramètre `dtype` accepte les types NumPy. `float32` occupe deux fois moins de mémoire que `float64`, au prix d'une précision moindre (environ 7 chiffres significatifs). C'est un choix fréquent pour les données destinées à des réseaux de neurones.

### À partir d'un dictionnaire

Les clés deviennent l'index, les valeurs deviennent les valeurs.

In [7]:
stock = pd.Series({"clavier": 12, "souris": 40, "ecran": 7}, name="prix")
stock

clavier    12
souris     40
ecran       7
Name: prix, dtype: int64

### À partir d'un tableau NumPy

In [6]:
s_np = pd.Series(np.arange(3), index=list("cde"))
s_np

c    0
d    1
e    2
dtype: int64

Vous rencontrerez aussi `from pandas import Series` dans du code existant. C'est équivalent, mais la convention `import pandas as pd` puis `pd.Series` est la plus répandue et la plus lisible.

## 2. Les attributs d'une Series

In [10]:
print("shape :", s.shape)
print("dtype :", s.dtype)
print("values:", s.values, type(s.values))
print("index :", s.index)

shape : (4,)
dtype : float64
values: [0.25 0.5  0.75 1.  ] <class 'numpy.ndarray'>
index : RangeIndex(start=0, stop=4, step=1)


- `values` renvoie le tableau NumPy sous-jacent : la Series **est** un tableau NumPy augmenté d'un index.
- `index` est un objet `Index`, immuable, qui contient les étiquettes.

Une Series se comporte aussi comme un dictionnaire : on peut la parcourir par paires (étiquette, valeur).

In [11]:
for etiquette, valeur in stock.items():
    print(etiquette, "→", valeur)

clavier → 12
souris → 40
ecran → 7


In [12]:
"souris" in stock     # le test porte sur l'index, comme pour les clés d'un dict

True

## 3. Le type des valeurs : nombres, texte, catégories

Pour du texte, Pandas 3 utilise le type `str`.

In [15]:
canaux = pd.Series(["web", "magasin", "web", "web", "telephone", "magasin"] * 1000)
canaux.dtype

<StringDtype(na_value=nan)>

Une colonne qui ne contient que quelques valeurs distinctes répétées peut être convertie en `category`. Pandas stocke alors une seule fois chaque modalité, et un petit entier par ligne qui pointe vers elle.

> **Comparaison.** La Series `canaux` contient 6 000 textes, mais seulement 3 valeurs distinctes. Laquelle des deux versions occupera le moins de mémoire ? Dans quelle proportion, selon vous ?

In [16]:
print("str      :", canaux.memory_usage(deep=True), "octets")
print("category :", canaux.astype("category").memory_usage(deep=True), "octets")

str      : 80132 octets
category : 6176 octets


In [17]:
canaux.astype("category").cat.categories, canaux.astype("category").cat.codes.head()

(Index(['magasin', 'telephone', 'web'], dtype='str'),
 0    2
 1    0
 2    2
 3    2
 4    1
 dtype: int8)

Les **codes** sont des entiers (`int8` ici) : 0 pour `magasin`, 1 pour `telephone`, 2 pour `web`. C'est exactement l'encodage dont un modèle de machine learning a besoin. Le chapitre 9 y reviendra.

Le choix du type entier compte aussi : `uint8` (entier non signé sur 8 bits, de 0 à 255) occupe 8 fois moins de place que `int64`.

In [18]:
quantites = pd.Series([1, 2, 1, 3, 5])
print(quantites.dtype, quantites.nbytes, "octets")
print(quantites.astype("uint8").dtype, quantites.astype("uint8").nbytes, "octets")

int64 40 octets
uint8 5 octets


## 4. Opérations vectorisées

Comme avec NumPy, les opérations s'appliquent à toutes les valeurs, sans boucle.

In [19]:
prix = pd.Series({"clavier": 89.9, "souris": 24.5, "ecran": 249.0})
prix * 1.21          # prix TVA comprise

clavier    108.779
souris      29.645
ecran      301.290
dtype: float64

In [20]:
prix > 50            # comparaison : une Series de booléens, avec le même index

clavier     True
souris     False
ecran       True
dtype: bool

Les fonctions universelles de NumPy (*ufuncs*) acceptent une Series et **conservent son index**.

In [ ]:
np.log(prix)

In [21]:
np.sum(prix), prix.sum()

(np.float64(363.4), np.float64(363.4))

## 5. L'alignement par l'index

C'est la différence fondamentale avec NumPy. Quand deux Series sont combinées, Pandas **aligne d'abord les étiquettes**, puis calcule.

In [23]:
stock_magasin = pd.Series({"clavier": 12, "souris": 40, "ecran": 7})
stock_entrepot = pd.Series({"souris": 100, "ecran": 20, "casque": 15})

clavier    12
souris     40
ecran       7
dtype: int64

> **Prédiction.** Que va produire `stock_magasin + stock_entrepot` ? Combien de lignes ? Quelle valeur pour `clavier` et pour `casque` ? Quel sera le type du résultat ?

In [24]:
stock_magasin + stock_entrepot

casque       NaN
clavier      NaN
ecran       27.0
souris     140.0
dtype: float64

Le résultat contient **l'union des étiquettes**, triée. Quand une étiquette n'existe que d'un côté, Pandas ne peut pas calculer : il renvoie `NaN` (*Not a Number*), la valeur manquante. Comme `NaN` est un flottant, le résultat passe de `int64` à `float64`.

Avec NumPy, la même opération aurait additionné les valeurs **position par position**, et donc mélangé souris et clavier sans aucune erreur.

In [ ]:
np.array([12, 40, 7]) + np.array([100, 20, 15])     # clavier+souris, souris+ecran, ecran+casque : absurde

### Remplacer les absents par une valeur : `fill_value`

Les méthodes `add`, `sub`, `mul` et `div` font la même chose que les opérateurs, avec un paramètre en plus : la valeur à utiliser quand une étiquette manque d'un côté.

In [30]:
stock_magasin.add(stock_entrepot, fill_value=0).astype(int)

casque      57
clavier     54
ecran       27
souris     140
dtype: int64

Le résultat reste en `float64` alors qu'il ne contient plus de `NaN`. L'alignement crée d'abord les valeurs manquantes (donc des flottants), puis `fill_value` les remplace. On peut revenir aux entiers avec `.astype(int)`.

### Division par zéro

In [31]:
pd.Series([10, 0, 5]) / pd.Series([2, 0, 0])

0    5.0
1    NaN
2    inf
dtype: float64

Pandas ne lève pas d'exception : `5 / 0` donne `inf` (l'infini) et `0 / 0` donne `NaN`, conformément à la norme des nombres flottants. Ces valeurs se propagent ensuite silencieusement dans les calculs : il faut savoir les repérer.

## 6. Modifier une valeur

In [41]:
stock.loc["souris"] = 35
stock

clavier    12
souris     35
ecran       7
casque      9
Name: prix, dtype: int64

In [45]:
stock.loc["casque"] = 9          # une étiquette inconnue ajoute une ligne
stock

clavier    12
souris     35
ecran       7
casque      9
Name: prix, dtype: int64

La syntaxe `stock["souris"] = 35` fonctionne aussi. Le chapitre 4 explique pourquoi `.loc` est préférable : il indique sans ambiguïté que l'on parle d'une étiquette.

## 7. Une Series filtrée reste une Series

In [48]:
filtre = prix[prix > 50]
type(filtre), type(filtre.values)

(pandas.Series, numpy.ndarray)

`prix[prix > 50]` conserve les étiquettes des lignes retenues. Pour récupérer un tableau NumPy brut (par exemple pour une bibliothèque de calcul), on utilise `.to_numpy()`, l'équivalent moderne et recommandé de `.values`.

In [49]:
filtre.to_numpy()

array([ 89.9, 249. ])

## Pièges fréquents

| Situation | Ce qui se passe | Réflexe |
|---|---|---|
| Additionner deux Series aux index différents | `NaN` pour chaque étiquette absente d'un côté | vérifier les index, ou utiliser `add(..., fill_value=0)` |
| Une colonne d'entiers contient un `NaN` | elle passe en `float64` | normal : `NaN` est un flottant. Le type `Int64` (avec majuscule) accepte les entiers manquants |
| Division par zéro | `inf` ou `NaN`, sans erreur | contrôler avec `np.isinf` et `isna` |
| Index avec doublons | une étiquette peut renvoyer plusieurs valeurs | vérifier `s.index.is_unique` |

In [ ]:
pd.Series([1, None, 3]).dtype, pd.Series([1, None, 3], dtype="Int64")

## Exercices

- Exercice supplémentaire **S1** : prédire le résultat d'opérations entre Series aux index décalés (`exercices/supplementaires/S01_alignement.ipynb`).

## À retenir

- Une Series = un tableau NumPy + un index + un nom éventuel.
- On la crée à partir d'une liste, d'un dictionnaire ou d'un tableau NumPy, avec `index=` et `dtype=` si besoin.
- Les opérations sont vectorisées et **alignées sur l'index** : les étiquettes absentes d'un côté produisent `NaN`.
- `add(..., fill_value=...)` remplace les absents avant le calcul.
- Le type choisi (`str`, `category`, `uint8`, `float32`) a un impact direct sur la mémoire.